In [10]:
from ultralytics import YOLO
import cv2
import numpy as np
from deepface import DeepFace
import json
import pandas as pd
import re
import time

In [11]:

def load_model():
    face_datector = YOLO("yolov11n-face.pt")
    face_regonizer = DeepFace
    return face_datector, face_regonizer


In [12]:

def Init_Camera():
    cap = cv2.VideoCapture(0)
    print("Camera is starting...")
    if not cap.isOpened():
        print("Cannot open camera")
        exit()
    return cap


In [13]:

def get_picture(camera):
    print("Getting frame from camera...")
    ret, frame = camera.read()
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        return None
    return frame

In [14]:
def close_Camera(camera):
    camera.release()
    cv2.destroyAllWindows()



In [15]:
def detect_face(face_datector, frame):
    results = face_datector(frame)
    boxes = results[0].boxes.xyxy.cpu().numpy()  # Extract bounding box coordinates
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)  # Convert coordinates to integers
        cropped_face = frame[y1-75:y2+75, x1-75:x2+75]  # Crop the face from the frame
    return cropped_face


In [16]:
def recognize_face(input_, database):
    try:
        # result = DeepFace.find(input_, database, normalization='ArcFace')
        result = DeepFace.find(input_, database)
        print(str(result))
        
        df = pd.DataFrame(result[0])
        record = json.loads(df.to_json(orient="records"))

        with open("match.json", "w") as f:
            for i in record:
                try:
                    if i['confidence'] >= 65:
                        f.write(json.dumps(i["identity"].strip("faceDB/"), indent=2))
                        f.write("\n")
                except Exception as e:
                    print("Error writing identity: ", str(e))
                    return False, None
            print("JSON File Saved")
        return True, record[0]
    
    except Exception as e:
        print("Some errors occurred: ", str(e))
        return


In [17]:
face_datector, face_regonizer = load_model()

In [39]:

start_time = time.time()
print("Models loaded.")
print("Initializing camera...")
camera = Init_Camera()
try:
        image = get_picture(camera)

        if image is not None:
            cv2.imwrite("picture_1.jpg", image)
            print(f"Picture 1 saved. Image shape: {image.shape}")
            results = detect_face(face_datector, image)
            cv2.imwrite("cropped_face.jpg", results)
            print(f"Cropped face saved. Cropped shape: {results.shape}")
            found, who = recognize_face(results, "faceDB")
            who = re.sub(r'(?i)^(?:real)?\s*(peem)\d*$', r'\1', who["identity"].strip("faceDB\\"))
            who = who[0:len(who)-5]
        if(found):
                print(f"Found face is database!\nHello nigga {(who)}")
        else:
                print("you're not my gng nigga😭🙏🥀")

except SystemExit as e:
        print(e)       
finally:
        print("Releasing camera...")
        close_Camera(camera)
end_time = time.time()
print(f"Total time taken: {end_time - start_time} seconds")

Models loaded.
Initializing camera...
Camera is starting...
Getting frame from camera...
Picture 1 saved. Image shape: (480, 640, 3)

0: 480x640 1 face, 14.4ms
Speed: 1.9ms preprocess, 14.4ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)
Cropped face saved. Cropped shape: (242, 216, 3)
25-11-03 21:44:27 - Searching [[[155 142 127]
  [159 144 122]
  [162 144 117]
  ...
  [169 173 178]
  [170 174 179]
  [171 175 179]]

 [[151 135 122]
  [158 139 119]
  [165 143 118]
  ...
  [169 175 178]
  [170 176 179]
  [172 177 180]]

 [[158 139 129]
  [166 142 126]
  [167 140 117]
  ...
  [167 174 176]
  [169 176 178]
  [171 177 179]]

 ...

 [[217 217 217]
  [212 212 212]
  [213 213 213]
  ...
  [149 163 188]
  [163 178 201]
  [166 182 202]]

 [[215 215 215]
  [212 212 212]
  [214 214 214]
  ...
  [134 151 175]
  [150 167 190]
  [164 182 203]]

 [[220 220 220]
  [215 215 215]
  [216 216 216]
  ...
  [121 139 163]
  [136 154 177]
  [157 176 198]]] in 3 length datastore
25-11-03 21: